In [1]:
import pandas as pd

df = pd.read_csv("../data/filteredvehpub.csv")   # adjust path as needed
df.head()
df.shape

(256115, 13)

In [15]:
df.columns

Index(['HOUSEID', 'HHFAMINC', 'HHSIZE', 'LIF_CYC', 'CENSUS_R', 'URBAN',
       'VEHID', 'DRVRCNT', 'MAKE', 'HH_RACE', 'WRKCOUNT', 'HOMEOWN',
       'URBANSIZE'],
      dtype='object')

In [17]:
import numpy as np
invalid_values = [-9, -8, -7, -88, 99, "99", "XX", "xx", "XX ", "-88", "-9", "-8", "-7"]
df = df.replace(invalid_values, np.nan)
df = df.dropna()

In [19]:
#df["VEHTYPE"] = df["VEHTYPE"].replace([5, 6], np.nan)
#df = df.dropna()

In [21]:
counts = df["MAKE"].value_counts()
counts.describe()

count       53.000000
mean      4619.924528
std       8184.317383
min         59.000000
25%        286.000000
50%       1584.000000
75%       4649.000000
max      34870.000000
Name: count, dtype: float64

In [23]:
df["MAKE"] = df["MAKE"].astype(str)   # convert everything to string first
df["MAKE"] = df["MAKE"].str.strip()   # remove whitespace
df["MAKE"] = df["MAKE"].astype(int)   # convert to integer

counts = df["MAKE"].value_counts()
rare_makes = counts[counts < 1000].index

df["MAKE"] = df["MAKE"].where(~df["MAKE"].isin(rare_makes), 98)
df["MAKE"].value_counts()

MAKE
12    34870
49    33679
20    31287
37    23895
7     12576
35    11706
98     8243
2      7309
48     6945
23     6915
55     6773
59     4980
18     4867
6      4713
63     4649
30     4622
34     4545
41     4340
42     3651
72     3094
19     2667
22     2665
54     2540
51     1832
14     1800
13     1789
24     1584
32     1495
58     1420
52     1240
53     1161
76     1004
Name: count, dtype: int64

In [25]:
#df.to_csv("processed_data.csv", index=False)

In [27]:
df.columns.tolist()

['HOUSEID',
 'HHFAMINC',
 'HHSIZE',
 'LIF_CYC',
 'CENSUS_R',
 'URBAN',
 'VEHID',
 'DRVRCNT',
 'MAKE',
 'HH_RACE',
 'WRKCOUNT',
 'HOMEOWN',
 'URBANSIZE']

In [29]:
feature_cols = [
    "HHSIZE",
    "HHFAMINC",
    "LIF_CYC",
    "CENSUS_R",
    "HH_RACE",
    "HOMEOWN",
    "WRKCOUNT",
    "URBAN",
    "URBANSIZE",
    "DRVRCNT"
]
target_col = "MAKE"
X = df[feature_cols]
y = df[target_col]

In [31]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler


numeric_features = ["HHSIZE", "HHFAMINC", "WRKCOUNT", "DRVRCNT"]
categorical_features = ["LIF_CYC", "CENSUS_R", "HH_RACE", "HOMEOWN", "URBAN", "URBANSIZE"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [33]:
import xgboost as xgb
from sklearn.pipeline import Pipeline

def make_xgb_pipeline():
    xgb_clf = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        eval_metric="mlogloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=42
    )
    pipe = Pipeline([
        ("preprocess", preprocess),
        ("xgb", xgb_clf)
    ])
    return pipe

In [35]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from tqdm import tqdm

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = []

for train_idx, test_idx in tqdm(cv.split(X, y), total=5):
    X_train_cv, X_test_cv = X.iloc[train_idx], X.iloc[test_idx]
    y_train_cv, y_test_cv = y.iloc[train_idx], y.iloc[test_idx]

    pipe_cv = make_xgb_pipeline()   # NEW pipeline for this fold
    pipe_cv.fit(X_train_cv, y_train_cv)

    y_pred_cv = pipe_cv.predict(X_test_cv)
    scores.append(accuracy_score(y_test_cv, y_pred_cv))

print("Cross-validated Accuracy:", np.mean(scores), "+/-", np.std(scores))

  0%|                                                     | 0/5 [00:00<?, ?it/s]


ValueError: Invalid classes inferred from unique values of `y`.  Expected: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31], got [ 2  6  7 12 13 14 18 19 20 22 23 24 30 32 34 35 37 41 42 48 49 51 52 53
 54 55 58 59 63 72 76 98]

In [ ]:
from sklearn.model_selection import train_test_split

# Train/test split (for final evaluation and confusion matrix)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Fit final XGBoost model on training data
pipe.fit(X_train, y_train)

# Predict on test set
from sklearn.metrics import accuracy_score, confusion_matrix
y_pred = pipe.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred))

# ================== 7. Confusion Matrix ==================
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)
labels = sorted(y.unique())

fig, ax = plt.subplots(figsize=(10, 10))
im = ax.imshow(cm, cmap="Blues")
plt.colorbar(im)

ax.set_xticks(np.arange(len(labels)))
ax.set_yticks(np.arange(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)

ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix (Top-1 Predictions, XGBoost)")

plt.tight_layout()
plt.show()

In [ ]:
pipe.fit(X_train, y_train)

In [ ]:
pipe.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

cm = confusion_matrix(y_test, y_pred)

labels = sorted(y.unique())

fig, ax = plt.subplots(figsize=(10, 10))
im = ax.imshow(cm, cmap="Blues")
plt.colorbar(im)

ax.set_xticks(np.arange(len(labels)))
ax.set_yticks(np.arange(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)

ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix (Top-1 Predictions)")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# 1. Define the XGBoost-specific parameter grid
# Note: We use 'clf__estimator__' because your pipeline has:
# Pipeline -> 'clf' (OneVsRest) -> 'estimator' (XGBoost)
param_dist = {
    'clf__estimator__n_estimators': [100, 300, 500],
    'clf__estimator__max_depth': [3, 5, 7],      # XGBoost prefers shallower trees than RF
    'clf__estimator__learning_rate': [0.01, 0.1, 0.2],
    'clf__estimator__subsample': [0.7, 0.8, 1.0], # Randomly sample rows to prevent overfitting
    'clf__estimator__colsample_bytree': [0.7, 0.8, 1.0] # Randomly sample columns
}

# 2. Setup the Search
# We use n_jobs=1 to avoid the Windows crash/hang you saw earlier
random_search = RandomizedSearchCV(
    estimator=pipe,     # This uses the 'pipe' defined in your Cell 13/14
    param_distributions=param_dist,
    n_iter=5,           # Tries 5 random combinations (Keep low for speed)
    cv=2,               # 2-fold cross-validation (Keep low for speed)
    scoring='f1_weighted',
    n_jobs=1,           # Crucial for Windows stability
    verbose=2,
    random_state=42
)

# 3. Run the experiment
print("Starting XGBoost Hyperparameter Tuning...")
random_search.fit(X_train, Y_train)

# 4. View Results
print(f"Test Complete. Best Params: {random_search.best_params_}")
print(f"Best Cross-Validation Score: {random_search.best_score_}")